### Ноутбук "EDA и подготовка данных"

#### Описание

Генерация и загрузка синтетических данных в PostgreSQL, первичная проверка целостности перед основным анализом.

##### Импорт модулей и библиотек

In [1]:
import sys
sys.path.append("../src")

from generate_data import generate_users, generate_subscription, generate_payments, generate_ab_assignments, generate_events

In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import text

from db_connection import get_engine
engine = get_engine()

#### Спринт 1 "Проектирование БД и генерация данных"

**Цель**: спроектировать реляционную схему для SaaS-продукта и наполнить её реалистичными синтетическими данными, чтобы было что анализировать.

##### Создание таблицы "users"

Генерация датафрейма "users_df" с 3000 пользователей, без столбца "id"

In [3]:
users_df = generate_users(3000)
display(users_df.head())
display(users_df.info())

,acquisition_channel,plan,country,signup_date
0,social,free,Russia,2026-07-11
1,organic,free,Spain,2025-09-20
2,organic,free,Russia,2026-02-13
3,paid_search,basic,Russia,2026-05-03
4,paid_search,pro,Russia,2026-06-29


<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   acquisition_channel  3000 non-null   str   
 1   plan                 3000 non-null   str   
 2   country              3000 non-null   str   
 3   signup_date          3000 non-null   object
dtypes: object(1), str(3)
memory usage: 93.9+ KB


None

*Очистка таблицы "users"*

In [4]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE users RESTART IDENTITY CASCADE;"))

Генерация датафрейма "users_db" со столбцом "id"

In [5]:
users_df.to_sql("users", engine, if_exists="append", index=False)

users_db = pd.read_sql("SELECT * FROM users;", engine)
display(users_db.columns)
display(users_db.shape)

Index(['id', 'acquisition_channel', 'plan', 'country', 'signup_date'], dtype='str')

(3000, 5)

##### Создание таблицы "subscription"

Генерация датафрейма "subscriptions_df" без столбца "id"

In [6]:
subscriptions_df = generate_subscription(users_db)
display(subscriptions_df.head())
display(subscriptions_df.info())

,user_id,plan,price,start_date,end_date,status
0,4,basic,100.0,2026-05-16,NaT,active
1,5,pro,250.0,2026-06-30,NaT,active
2,6,pro,250.0,2026-07-04,NaT,active
3,7,pro,250.0,2025-09-14,2026-02-25,canceled
4,8,basic,100.0,2026-05-18,NaT,active


<class 'pandas.DataFrame'>
RangeIndex: 1347 entries, 0 to 1346
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   user_id     1347 non-null   int64         
 1   plan        1347 non-null   str           
 2   price       1347 non-null   float64       
 3   start_date  1347 non-null   datetime64[us]
 4   end_date    398 non-null    datetime64[us]
 5   status      1347 non-null   str           
dtypes: datetime64[us](2), float64(1), int64(1), str(2)
memory usage: 63.3 KB


None

*Очистка таблицы "subscriptions"*

In [7]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE subscriptions RESTART IDENTITY CASCADE;"))

Генерация датафрейма "subscriptions_db" со столбцом "id"

In [8]:
subscriptions_df.to_sql("subscriptions", engine, if_exists="append", index=False)

subscriptions_db = pd.read_sql("SELECT * FROM subscriptions;", engine)
display(subscriptions_db.columns)
display(subscriptions_db.shape)

Index(['id', 'user_id', 'plan', 'price', 'start_date', 'end_date', 'status'], dtype='str')

(1347, 7)

##### Создание таблицы "payments"

Генерация датафрейма "payments_df" без столбца "id"

In [9]:
payments_df = generate_payments(subscriptions_db)
display(payments_df.head())
display(payments_df.info())

,subscription_id,amount,payment_date,status
0,1,100.0,2026-05-16,succeeded
1,1,100.0,2026-06-15,succeeded
2,1,100.0,2026-07-15,succeeded
3,1,100.0,2026-08-14,succeeded
4,1,100.0,2026-09-13,succeeded


<class 'pandas.DataFrame'>
RangeIndex: 11096 entries, 0 to 11095
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   subscription_id  11096 non-null  int64         
 1   amount           11096 non-null  float64       
 2   payment_date     11096 non-null  datetime64[us]
 3   status           11096 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(1), str(1)
memory usage: 346.9 KB


None

*Очистка таблицы "payments"*

In [10]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE payments RESTART IDENTITY CASCADE;"))

Генерация датафрейма "payments_db" со столбцом "id"

In [11]:
payments_df.to_sql("payments", engine, if_exists="append", index=False)

payments_db = pd.read_sql("SELECT * FROM payments;", engine)
display(payments_db.columns)
display(payments_db.shape)

Index(['id', 'subscription_id', 'amount', 'payment_date', 'status'], dtype='str')

(11096, 5)

##### Создание таблицы "ab_test_assignments"

Генерация датафрейма "assignments_df" без столбца "id"

In [12]:
assignments_df = generate_ab_assignments(users_db)
display(assignments_df.head())
display(assignments_df.info())

,user_id,test,variant,assigned_at
0,1,onboarding,A,2026-07-14
1,1,pricing,B,2026-07-12
2,5,onboarding,A,2026-06-30
3,5,pricing,B,2026-07-02
4,6,onboarding,B,2026-07-01


<class 'pandas.DataFrame'>
RangeIndex: 1040 entries, 0 to 1039
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   user_id      1040 non-null   int64         
 1   test         1040 non-null   str           
 2   variant      1040 non-null   str           
 3   assigned_at  1040 non-null   datetime64[us]
dtypes: datetime64[us](1), int64(1), str(2)
memory usage: 32.6 KB


None

*Очистка таблицы "ab_test_assignments"*

In [13]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE ab_test_assignments RESTART IDENTITY CASCADE;"))

Генерация датафрейма "assignments_db" со столбцом "id"

In [14]:
assignments_df.to_sql("ab_test_assignments", engine, if_exists="append", index=False)

assignments_db = pd.read_sql("SELECT * FROM ab_test_assignments;", engine)
display(assignments_db.columns)
display(assignments_db.shape)

Index(['id', 'user_id', 'test', 'variant', 'assigned_at'], dtype='str')

(1040, 5)

In [15]:
for table in ['users', 'subscriptions', 'payments', 'events', 'ab_test_assignments']:
    count = pd.read_sql(f"SELECT COUNT(*) FROM {table};", engine)
    print(table, count.iloc[0,0])

users 3000
subscriptions 1347
payments 11096
events 0
ab_test_assignments 1040


##### Создание таблицы "events"

Генерация датафрейма "events_df" без столбца "id"

In [16]:
events_df = generate_events(users_db)
display(events_df.head())
display(events_df.info())

,user_id,event_type,event_time
0,1,signup,2026-07-11 00:00:00
1,1,onboarding_done,2026-07-11 05:00:00
2,2,signup,2025-09-20 00:00:00
3,2,onboarding_done,2025-09-22 01:00:00
4,3,signup,2026-02-13 00:00:00


<class 'pandas.DataFrame'>
RangeIndex: 6794 entries, 0 to 6793
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   user_id     6794 non-null   int64         
 1   event_type  6794 non-null   str           
 2   event_time  6794 non-null   datetime64[us]
dtypes: datetime64[us](1), int64(1), str(1)
memory usage: 159.4 KB


None

*Очистка таблицы "events"*

In [17]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE events RESTART IDENTITY CASCADE;"))

Генерация датафрейма "events_db" со столбцом "id"

In [18]:
events_df.to_sql("events", engine, if_exists="append", index=False)

events_db = pd.read_sql("SELECT * FROM events", engine)
display(events_db.columns)
display(events_db.shape)

Index(['id', 'user_id', 'event_type', 'event_time'], dtype='str')

(6794, 4)

**Вывод**: спроектирована и создана схема из 5 связанных таблиц в PostgreSQL (users, subscriptions, payments, events, ab_test_assignments), с осознанными архитектурными решениями (ENUM vs VARCHAR, SCD Type 2, нормализация). Написан generate_data.py, сгенерировано и загружено ~22 500 строк с заложенными реалистичными закономерностями (канал→конверсия, каскадная воронка). Настроено рабочее окружение (Python/venv, DBeaver, git/GitHub) с решением двух серьёзных инфраструктурных багов.

#### Спринт 2 "SQL-аналитика"

**Цель**: прокачать SQL на реальных аналитических задачах — воронка, когорты, оконные функции — с проверкой результатов через pandas.

Задача 5. Сверка помесячной выручки SQL vs pandas

In [19]:
df_success_payment = payments_db[payments_db["status"] == "succeeded"].copy()
df_success_payment["payment_date"] = pd.to_datetime(df_success_payment["payment_date"])
df_success_payment["payment_month"] = df_success_payment["payment_date"].dt.to_period("M")
df_monthly_revenue = df_success_payment.groupby("payment_month")["amount"].sum().reset_index()
df_monthly_revenue

,payment_month,amount
0,2025-04,10950.0
1,2025-05,23400.0
2,2025-06,33050.0
3,2025-07,43850.0
4,2025-08,54200.0
5,2025-09,63700.0
6,2025-10,77300.0
7,2025-11,84300.0
8,2025-12,96200.0
9,2026-01,99100.0


**Вывод**: реализованы и сохранены в sql/analysis_queries.sql пять запросов: воронка по шагам, конверсия между шагами (LAG()), когортный retention (сначала упрощённый, потом полноценный по возрасту когорты через CTE + AGE()/EXTRACT()), помесячная выручка с накопительным итогом (SUM() OVER). Подтверждена идентичность результатов SQL и pandas на одном срезе данных.

#### Спринт 3. Pandas — аудит, очистка и кодирование данных

**Цель**: на практическом кейсе (намеренно "испорченные" данные с пропусками, дублями, разнотипными аномалиями и рассинхроном между таблицами) освоить полный рабочий цикл дата-аналитика в pandas — от аудита до готовых к анализу данных, включая merge/join, groupby+transform, pivot_table, обработку пропусков/дублей/выбросов и кодирование категорий — как фундамент для следующих спринтов (визуализация, статистика, A/B-тесты).

**Вывод:**